# CH14 — Grafos (parte 3): aplicaciones. Caminos más cortos con Dijkstra

Material del curso basado en Goodrich, Tamassia & Goldwasser — Capítulo 14 (sección 14.6).

**Objetivo:** encontrar la ruta de **menor costo** en un grafo con **pesos**, que es el problema que resuelven aplicaciones como Waze o Google Maps. Entenderemos por qué BFS ya no alcanza, cómo funciona el algoritmo de **Dijkstra** y cómo usar el código original de Goodrich (`goodrich/ch14`).

**Contenido**

0. Preparación: importar el código de Goodrich
1. El problema: BFS no mira los pesos
2. La idea de Dijkstra
3. Dijkstra paso a paso
4. El código de Goodrich
5. ¿Cuánto cuesta?
6. Dos trampas: vértices inalcanzables y pesos negativos
7. Aplicación: la ruta más rápida (Waze)
8. Aplicación: cruzar un terreno con barro
9. Práctica: Dijkstra a mano
10. Resumen y ejercicios

## 0. Preparación: importar el código de Goodrich

La celda siguiente busca la carpeta `goodrich` subiendo desde el directorio del cuaderno y la agrega a `sys.path`, así funciona en cualquier computador sin escribir rutas fijas.

| Archivo | Qué trae |
|---|---|
| `goodrich/ch14/graph.py` | la clase `Graph` |
| `goodrich/ch14/bfs.py` | `BFS(g, s, discovered)` |
| `goodrich/ch14/dfs.py` | `construct_path(u, v, discovered)` |
| `goodrich/ch14/shortest_paths.py` | `shortest_path_lengths(g, src)` y `shortest_path_tree(g, s, d)` |
| `goodrich/ch09/adaptable_heap_priority_queue.py` | la cola de prioridad que usa Dijkstra |

In [3]:
import sys
from pathlib import Path

raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'goodrich').is_dir())
sys.path.append(str(raiz))

In [4]:
import inspect

from goodrich.ch09.adaptable_heap_priority_queue import AdaptableHeapPriorityQueue
from goodrich.ch14.bfs import BFS
from goodrich.ch14.dfs import construct_path
from goodrich.ch14.graph import Graph
from goodrich.ch14.shortest_paths import shortest_path_lengths, shortest_path_tree

El **peso** de una arista es su elemento: `e.element()`. La función `construir` arma un `Graph` a partir de tuplas `(origen, destino, peso)`, conservando el orden de inserción, y retorna también el diccionario `etiqueta -> Vertex`.

Usamos el grafo ponderado que ya presentamos en el cuaderno de teoría.

In [5]:
def construir(aristas, dirigido=False, aislados=()):
    # Construye un Graph con pesos. Retorna (g, V) con V: etiqueta -> Vertex.
    g = Graph(dirigido)
    V = {}
    for u, v, _ in aristas:
        for x in (u, v):
            if x not in V:
                V[x] = g.insert_vertex(x)
    for x in aislados:
        V[x] = g.insert_vertex(x)
    for u, v, peso in aristas:
        g.insert_edge(V[u], V[v], peso)
    return g, V


PONDERADO = [('A', 'B', 4), ('A', 'C', 2), ('B', 'C', 1), ('B', 'D', 5), ('C', 'D', 8),
             ('C', 'E', 10), ('D', 'E', 2), ('D', 'F', 6), ('E', 'F', 3)]
g, V = construir(PONDERADO)

print(g.vertex_count(), 'vértices,', g.edge_count(), 'aristas')
print('aristas de D:', list(g.incident_edges(V['D'])))

6 vértices, 9 aristas
aristas de D: [(B,D,5), (C,D,8), (D,E,2), (D,F,6)]


## 1. El problema: BFS no mira los pesos

En el cuaderno anterior, BFS encontraba el camino con **menos aristas**. Eso es lo mismo que el camino más corto solo si todas las aristas valen lo mismo.

En un grafo **ponderado**, el **costo** de un camino es la **suma de los pesos** de sus aristas. Un camino con más aristas puede salir más barato: dos calles despejadas pueden tomar menos tiempo que una sola con trancón.

Comparemos BFS con la mejor ruta posible de `A` a `D`:

In [6]:
def costo(g, camino):
    # Suma de los pesos de las aristas de un camino (lista de Vertex).
    return sum(g.get_edge(u, v).element() for u, v in zip(camino, camino[1:]))


discovered = {V['A']: None}
BFS(g, V['A'], discovered)
camino_bfs = construct_path(V['A'], V['D'], discovered)

print('BFS:          ', camino_bfs, ' costo', costo(g, camino_bfs))

mejor = [V['A'], V['C'], V['B'], V['D']]
print('otro camino:  ', mejor, ' costo', costo(g, mejor))

BFS:           [A, B, D]  costo 9
otro camino:   [A, C, B, D]  costo 8


BFS entrega un camino de 2 aristas con costo 9, pero existe uno de 3 aristas con costo 8. BFS no puede verlo porque **nunca lee los pesos**.

Probar todos los caminos tampoco sirve: su número puede crecer exponencialmente con el tamaño del grafo. Necesitamos otro algoritmo.

## 2. La idea de Dijkstra

Dijkstra se parece a BFS: también avanza como una onda desde el inicio $s$. La diferencia es que la onda crece por **costo acumulado**, no por número de aristas.

Para cada vértice $v$ guardamos $d[v]$, el **costo del mejor camino conocido hasta ahora** de $s$ a $v$. Al principio $d[s] = 0$ y $d[v] = \infty$ para todos los demás. Además mantenemos una **nube**: los vértices cuya distancia ya es definitiva.

Se repite lo siguiente hasta que no queden vértices:

1. **Elegir:** sacar el vértice $u$ que **no** está en la nube y tiene el menor $d[u]$.
2. **Fijar:** meter $u$ a la nube. Su distancia ya no cambiará.
3. **Relajar:** para cada arista $(u, v)$ con $v$ fuera de la nube, revisar si pasar por $u$ mejora el camino a $v$:

$$\text{si } d[u] + w(u, v) < d[v] \quad \text{entonces} \quad d[v] \leftarrow d[u] + w(u, v)$$

Es un algoritmo **voraz**: en cada paso toma la mejor opción disponible y nunca se devuelve.

**¿Por qué se puede fijar $u$ con tanta seguridad?** Cualquier otro camino hacia $u$ tendría que salir de la nube por algún vértice $x$ con $d[x] \geq d[u]$, porque $u$ era el mínimo. Si los pesos no son negativos, seguir de $x$ hasta $u$ no puede abaratar ese camino, así que nunca costará menos que $d[u]$.

Esta razón depende de que **ningún peso sea negativo**. En la sección 6 veremos qué pasa si lo es.

## 3. Dijkstra paso a paso

Cada panel muestra el estado **después** de un paso, con $d[v]$ debajo de cada letra:

![Dijkstra paso a paso desde A](../assets/ch14_dijkstra_pasos.png)

Observa dos momentos:

- **Paso 2:** al salir `C`, la distancia de `B` baja de 4 a 3. La arista directa `A–B` deja de ser la mejor opción, porque `A → C → B` cuesta menos.
- **Paso 3:** al salir `B`, la distancia de `D` baja de 10 a 8. Este es el camino de 3 aristas que BFS no podía ver.

La función de abajo es **el mismo algoritmo** de Goodrich, con una sola diferencia: va imprimiendo una fila por cada paso. Las distancias entre `[ ]` son las que ya están en la nube.

In [7]:
def dijkstra_traza(g, s):
    # Igual a shortest_path_lengths de Goodrich, pero imprime cada paso.
    d = {}
    cloud = {}
    pq = AdaptableHeapPriorityQueue()
    pqlocator = {}
    for v in g.vertices():
        d[v] = 0 if v is s else float('inf')
        pqlocator[v] = pq.add(d[v], v)

    print('paso  sale   ' + ''.join(f'{str(v):>7}' for v in g.vertices()))
    paso = 0
    while not pq.is_empty():
        key, u = pq.remove_min()
        cloud[u] = key
        del pqlocator[u]
        for e in g.incident_edges(u):
            v = e.opposite(u)
            if v not in cloud:
                wgt = e.element()
                if d[u] + wgt < d[v]:
                    d[v] = d[u] + wgt
                    pq.update(pqlocator[v], d[v], v)
        paso += 1
        fila = ''
        for v in g.vertices():
            texto = '∞' if d[v] == float('inf') else str(d[v])
            fila += f'{"[" + texto + "]" if v in cloud else texto:>7}'
        print(f'{paso:>4}  {str(u):>4}   {fila}')
    return cloud


_ = dijkstra_traza(g, V['A'])

paso  sale         A      B      C      D      E      F
   1     A       [0]      4      2      ∞      ∞      ∞
   2     C       [0]      3    [2]     10     12      ∞
   3     B       [0]    [3]    [2]      8     12      ∞
   4     D       [0]    [3]    [2]    [8]     10     14
   5     E       [0]    [3]    [2]    [8]   [10]     13
   6     F       [0]    [3]    [2]    [8]   [10]   [13]


## 4. El código de Goodrich

Goodrich separa el trabajo en dos funciones: una calcula las **distancias** y la otra reconstruye los **caminos**.

### 4.1 `shortest_path_lengths`: las distancias

Elegir "el vértice con menor $d$" es exactamente lo que hace una **cola de prioridad**, como las del capítulo 9. Pero hay un detalle: cuando relajamos una arista, la distancia de un vértice que **ya está en la cola** cambia.

Por eso usa `AdaptableHeapPriorityQueue`. Al agregar un elemento, `pq.add(...)` retorna un **localizador**, y con `pq.update(localizador, nueva_clave, valor)` se cambia su clave sin tener que buscarlo. El diccionario `pqlocator` guarda el localizador de cada vértice.

In [8]:
print(inspect.getsource(shortest_path_lengths))

def shortest_path_lengths(g, src):
  """Compute shortest-path distances from src to reachable vertices of g.

  Graph g can be undirected or directed, but must be weighted such that
  e.element() returns a numeric weight for each edge e.

  Return dictionary mapping each reachable vertex to its distance from src.
  """
  d = {}                                        # d[v] is upper bound from s to v
  cloud = {}                                    # map reachable v to its d[v] value
  pq = AdaptableHeapPriorityQueue()             # vertex v will have key d[v]
  pqlocator = {}                                # map from vertex to its pq locator

  # for each vertex v of the graph, add an entry to the priority queue, with
  # the source having distance 0 and all others having infinite distance
  for v in g.vertices():
    if v is src:
      d[v] = 0
    else:
      d[v] = float('inf')                       # syntax for positive infinity
    pqlocator[v] = pq.add(d[v], v)              # save

In [9]:
d = shortest_path_lengths(g, V['A'])

for v, distancia in d.items():
    print(f'd[{v}] = {distancia}')

d[A] = 0
d[C] = 2
d[B] = 3
d[D] = 8
d[E] = 10
d[F] = 13


Coinciden con la última fila de la traza y con esta figura, donde las aristas rojas forman el **árbol de caminos más cortos**:

![Árbol de caminos más cortos desde A](../assets/ch14_dijkstra.png)

### 4.2 `shortest_path_tree`: los caminos

`shortest_path_lengths` solo retorna números. Para saber **por dónde** ir, `shortest_path_tree(g, s, d)` revisa cada vértice $v$ y busca una arista entrante $(u, v)$ que cumpla $d[v] = d[u] + w(u, v)$. Esa arista es por donde llega el camino más corto.

El resultado es un diccionario `vértice -> arista`, igual al `discovered` de BFS, salvo que `s` **no** aparece en él. Por eso podemos reconstruir el camino con la misma `construct_path` del cuaderno anterior. Su único caso especial es pedir el camino de `s` a `s`, para el que retorna `[]`.

In [8]:
print(inspect.getsource(shortest_path_tree))

def shortest_path_tree(g, s, d):
  """Reconstruct shortest-path tree rooted at vertex s, given distance map d.

  Return tree as a map from each reachable vertex v (other than s) to the
  edge e=(u,v) that is used to reach v from its parent u in the tree.
  """
  tree = {}
  for v in d:
    if v is not s:
      for e in g.incident_edges(v, False):       # consider INCOMING edges
        u = e.opposite(v)
        wgt = e.element()
        if d[v] == d[u] + wgt:
          tree[v] = e                            # edge e is used to reach v
  return tree



In [9]:
tree = shortest_path_tree(g, V['A'], d)

for x in 'BCDEF':
    r = construct_path(V['A'], V[x], tree)
    print(f'A -> {x}: costo {d[V[x]]:>2}   {r}')

A -> B: costo  3   [A, C, B]
A -> C: costo  2   [A, C]
A -> D: costo  8   [A, C, B, D]
A -> E: costo 10   [A, C, B, D, E]
A -> F: costo 13   [A, C, B, D, E, F]


El camino a `D` es `[A, C, B, D]`, con costo 8, el mismo que encontramos en la sección 1.

## 5. ¿Cuánto cuesta?

Sea $n$ el número de vértices y $m$ el de aristas. Dijkstra hace estas operaciones sobre la cola:

| Operación | Cuántas veces | Costo con un heap |
|---|---|---|
| `pq.add` | $n$ | $O(\log n)$ |
| `pq.remove_min` | $n$ | $O(\log n)$ |
| `pq.update` | a lo sumo una por cada revisión de arista, $O(m)$ | $O(\log n)$ |

El total es $O((n + m) \log n)$. Si la cola fuera una lista sin ordenar, cada `remove_min` costaría $O(n)$ y el total sería $O(n^2)$.

| | Heap: $O((n + m) \log n)$ | Lista sin ordenar: $O(n^2)$ |
|---|---|---|
| Grafo disperso, $m \approx n$ (un mapa de calles) | **mejor** | peor |
| Grafo muy denso, $m \approx n^2$ | $O(n^2 \log n)$ | **mejor** |

Los mapas de calles son dispersos, porque cada esquina tiene pocas calles. Por eso el heap es la opción natural en este tipo de aplicación.

## 6. Dos trampas

### 6.1 Vértices inalcanzables

El docstring de `shortest_path_lengths` dice que el resultado solo incluye los vértices alcanzables. Probémoslo con un grafo que tiene dos partes separadas:

In [10]:
g2, V2 = construir([('A', 'B', 3), ('B', 'C', 4), ('X', 'Y', 1)])

d2 = shortest_path_lengths(g2, V2['A'])
print(d2)

{A: 0, B: 3, C: 7, X: inf, Y: inf}


**El código no hace lo que dice su docstring:** `X` y `Y` aparecen con distancia `inf`. Todos los vértices entran a la cola, incluidos los inalcanzables, y al salir se guardan en `cloud`.

Esto tiene una consecuencia seria. Si le pasamos ese `d` a `shortest_path_tree`, se cumple `inf == inf + 1`, así que `X` y `Y` entran al árbol apuntándose **el uno al otro**. Si pidiéramos `construct_path(A, X, ...)`, nunca llegaría a `A` y se quedaría en un **ciclo infinito**, así que aquí solo imprimimos el árbol:

In [11]:
arbol_malo = shortest_path_tree(g2, V2['A'], d2)
print('árbol sin filtrar:', arbol_malo)

árbol sin filtrar: {B: (A,B,3), C: (B,C,4), X: (X,Y,1), Y: (X,Y,1)}


La solución es quitar las distancias infinitas **antes** de construir el árbol. Esta función reúne los dos pasos y es la que usaremos de aquí en adelante:

In [12]:
def caminos_mas_cortos(g, s):
    # Retorna (d, tree) solo con los vértices alcanzables desde s.
    d = {v: x for v, x in shortest_path_lengths(g, s).items() if x != float('inf')}
    return d, shortest_path_tree(g, s, d)


d2, tree2 = caminos_mas_cortos(g2, V2['A'])
print('distancias:', d2)
print('árbol:     ', tree2)
print('A -> C:', construct_path(V2['A'], V2['C'], tree2))
print('A -> X:', construct_path(V2['A'], V2['X'], tree2), ' (no hay camino)')

distancias: {A: 0, B: 3, C: 7}
árbol:      {B: (A,B,3), C: (B,C,4)}
A -> C: [A, B, C]
A -> X: []  (no hay camino)


### 6.2 Pesos negativos

Dijkstra **no funciona** con pesos negativos, porque deja de ser cierto que una distancia fijada no puede mejorar. Lo vemos en este grafo dirigido, donde el mejor camino a `B` es `A → C → B`, con costo $3 + (-2) = 1$:

In [13]:
g3, V3 = construir([('A', 'B', 2), ('A', 'C', 3), ('C', 'B', -2)], dirigido=True)

d3 = shortest_path_lengths(g3, V3['A'])
print('Dijkstra dice que d[B] =', d3[V3['B']])
print('el camino A -> C -> B cuesta', costo(g3, [V3['A'], V3['C'], V3['B']]))

Dijkstra dice que d[B] = 2
el camino A -> C -> B cuesta 1


`B` sale de la cola con distancia 2, antes que `C`, y queda fijado. Cuando después sale `C`, la arista `C → B` ni siquiera se revisa, porque `B` ya está en la nube. Dijkstra no da ningún error: simplemente entrega una **respuesta equivocada**.

Para grafos con pesos negativos existen otros algoritmos, como Bellman-Ford, que no veremos en este curso. En tiempos y distancias este problema no suele aparecer, porque esos valores no son negativos.

## 7. Aplicación: la ruta más rápida (Waze)

Volvamos a la figura del cuaderno de teoría. Los vértices son esquinas, las aristas son calles y cada peso es el tiempo en minutos.

![Aplicaciones de los grafos](../assets/ch14_aplicaciones.png)

Queremos ir de la casa a la universidad. A la vista, la calle `A–C` parece un buen atajo, pero tiene un trancón de 25 minutos.

In [14]:
CALLES = {('Casa', 'A'): 7, ('Casa', 'B'): 4, ('A', 'B'): 2, ('A', 'C'): 25, ('B', 'D'): 6,
          ('C', 'D'): 8, ('C', 'U'): 3, ('D', 'U'): 9}


def ruta(calles, origen, destino):
    # Calcula e imprime la ruta más rápida con Dijkstra.
    g, V = construir([(u, v, minutos) for (u, v), minutos in calles.items()])
    d, tree = caminos_mas_cortos(g, V[origen])
    print(' -> '.join(str(v) for v in construct_path(V[origen], V[destino], tree)), f'  ({d[V[destino]]} min)')


ruta(CALLES, 'Casa', 'U')

Casa -> B -> D -> U   (19 min)


Lo que hace especial a Waze es que **los pesos cambian**. Si los usuarios reportan un accidente, el peso de esa arista sube y la ruta se recalcula. La estructura del grafo sigue igual: solo cambian los números.

Veamos tres situaciones:

In [15]:
print('accidente en D-U (30 min):')
ruta({**CALLES, ('D', 'U'): 30}, 'Casa', 'U')

print('\nse despeja A-C (5 min):')
ruta({**CALLES, ('A', 'C'): 5}, 'Casa', 'U')

print('\ncierran Casa-B (ya no existe la calle):')
sin_casa_b = {calle: minutos for calle, minutos in CALLES.items() if calle != ('Casa', 'B')}
ruta(sin_casa_b, 'Casa', 'U')

accidente en D-U (30 min):
Casa -> B -> D -> C -> U   (21 min)

se despeja A-C (5 min):
Casa -> B -> A -> C -> U   (14 min)

cierran Casa-B (ya no existe la calle):
Casa -> A -> B -> D -> U   (24 min)


Las aplicaciones reales usan variantes más rápidas de esta idea, porque un mapa tiene millones de esquinas: A\*, preprocesamiento del mapa, etc. Pero la base es la misma.

## 8. Aplicación: cruzar un terreno con barro

En el cuaderno anterior, BFS encontraba la salida de un laberinto con el menor número de **pasos**. Ahora cambiemos el problema: algunas casillas son de **barro** (`~`) y entrar a ellas cuesta 5, mientras que una casilla normal cuesta 1.

Como el costo depende de la casilla a la que se **entra**, ir de una casilla normal al barro no cuesta lo mismo que salir de él. Por eso el modelo es un grafo **dirigido**, con una arista en cada sentido.

In [16]:
TERRENO = [
    '###########',
    '#E~~~~~~~S#',
    '#.#######.#',
    '#.........#',
    '###########',
]
COSTO_CASILLA = {'E': 1, 'S': 1, '.': 1, '~': 5}


def terreno_a_grafo(mapa):
    # Retorna (g, V): un Graph dirigido y el dict (fila, col) -> Vertex.
    g = Graph(directed=True)
    V = {}
    for f, linea in enumerate(mapa):
        for c, ch in enumerate(linea):
            if ch != '#':
                V[(f, c)] = g.insert_vertex((f, c))
    for (f, c), v in V.items():
        for vecino in ((f, c + 1), (f + 1, c)):               # derecha y abajo
            if vecino in V:
                w = V[vecino]
                g.insert_edge(v, w, COSTO_CASILLA[mapa[vecino[0]][vecino[1]]])   # entrar a w
                g.insert_edge(w, v, COSTO_CASILLA[mapa[f][c]])                   # entrar a v
    return g, V


def buscar(mapa, ch):
    # (fila, col) de la primera aparición del carácter ch.
    for f, linea in enumerate(mapa):
        c = linea.find(ch)
        if c != -1:
            return (f, c)


def dibujar(mapa, ruta):
    # Imprime el terreno marcando con 'o' las casillas de la ruta.
    celdas = {v.element() for v in ruta}
    for f, linea in enumerate(mapa):
        print(''.join('o' if (f, c) in celdas and ch in '.~' else ch for c, ch in enumerate(linea)))


t, VT = terreno_a_grafo(TERRENO)
entrada, salida = VT[buscar(TERRENO, 'E')], VT[buscar(TERRENO, 'S')]

discovered = {entrada: None}
BFS(t, entrada, discovered)
ruta_bfs = construct_path(entrada, salida, discovered)

dT, treeT = caminos_mas_cortos(t, entrada)
ruta_dijkstra = construct_path(entrada, salida, treeT)

print(f'BFS: {len(ruta_bfs) - 1} pasos, costo {costo(t, ruta_bfs)}')
dibujar(TERRENO, ruta_bfs)
print()
print(f'Dijkstra: {len(ruta_dijkstra) - 1} pasos, costo {costo(t, ruta_dijkstra)}')
dibujar(TERRENO, ruta_dijkstra)

BFS: 8 pasos, costo 36
###########
#EoooooooS#
#.#######.#
#.........#
###########

Dijkstra: 12 pasos, costo 12
###########
#E~~~~~~~S#
#o#######o#
#ooooooooo#
###########


BFS atraviesa el barro porque es el camino con menos pasos. Dijkstra da más pasos, pero su ruta cuesta mucho menos.

**¿Y si todas las casillas costaran lo mismo?** Con todos los pesos iguales a 1, el costo de un camino es su número de aristas, así que Dijkstra y BFS dan las **mismas distancias**. En ese caso conviene BFS, que es más simple y cuesta $O(n + m)$.

| | BFS | Dijkstra |
|---|---|---|
| Minimiza | número de aristas | suma de pesos |
| Pesos | los ignora | no negativos |
| Estructura | lista de nivel o cola | cola de prioridad adaptable |
| Costo | $O(n + m)$ | $O((n + m) \log n)$ |

## 9. Práctica: Dijkstra a mano

Haz Dijkstra **en papel** en este grafo, empezando en el vértice `0`. Dibuja una tabla con una fila por paso, como la de la sección 3, y anota qué vértice sale en cada una.

![Grafo para practicar Dijkstra a mano](../assets/ch14_dijkstra_a_mano.png)

Cuando termines, escribe tus distancias finales en `mi_tabla`, cambiando cada `None`, y ejecuta la celda. Solo te dice cuántas tienes bien, sin mostrarte la respuesta.

In [17]:
A_MANO = [(0, 1, 5), (0, 3, 7), (0, 4, 2), (1, 2, 7), (1, 5, 2), (1, 6, 5),
          (2, 5, 5), (3, 5, 7), (4, 6, 8), (5, 7, 3)]
gm, Vm = construir(A_MANO)

mi_tabla = {0: None, 1: None, 2: None, 3: None, 4: None, 5: None, 6: None, 7: None}

dm = shortest_path_lengths(gm, Vm[0])
if None in mi_tabla.values():
    print('Te faltan', list(mi_tabla.values()).count(None), 'distancias por escribir.')
else:
    bien = sum(mi_tabla[x] == dm[Vm[x]] for x in mi_tabla)
    print(f'{bien} de {len(mi_tabla)} correctas')

Te faltan 8 distancias por escribir.


## 10. Resumen y ejercicios

- En un grafo **ponderado**, el costo de un camino es la suma de sus pesos. BFS no sirve para minimizarlo, porque ignora los pesos.
- **Dijkstra** saca siempre el vértice fuera de la nube con menor $d$, lo fija y **relaja** sus aristas. Requiere pesos **no negativos**.
- Con `AdaptableHeapPriorityQueue` cuesta $O((n + m) \log n)$. La cola debe ser adaptable porque las distancias de los vértices cambian mientras están en ella.
- `shortest_path_lengths` da las distancias y `shortest_path_tree` el árbol de caminos.
- **Cuidado:** a pesar de su docstring, `shortest_path_lengths` incluye los vértices inalcanzables con distancia `inf`. Hay que quitarlos antes de llamar a `shortest_path_tree`.
- Modelar es la parte importante: decidir qué son los vértices, las aristas y los pesos (esquinas y minutos, casillas y costo de entrar).

**Patrón para usar el código de Goodrich:**

```python
d = {v: x for v, x in shortest_path_lengths(g, s).items() if x != float('inf')}
tree = shortest_path_tree(g, s, d)
d[v]                          # costo del camino más corto de s a v (si v in d)
construct_path(s, v, tree)    # los vértices de ese camino
```

**Ejercicios**

1. Modifica `ruta` para que, si no hay camino entre `origen` y `destino`, imprima un mensaje en vez de fallar.
2. Agrega la calle `('A', 'U')` con 12 minutos a `CALLES`. ¿Cambia la ruta más rápida desde `Casa`? Responde primero sin ejecutar.
3. Queremos la ruta entre `Casa` y `U` que **pase por** `C`. ¿Cómo la calcularías con dos llamadas a Dijkstra?
4. En `TERRENO`, ¿hasta qué costo de barro le conviene a Dijkstra cruzarlo? Prueba cambiando `COSTO_CASILLA['~']` y explica el resultado.
5. Escribe una función que retorne la lista de vértices a los que se puede llegar desde `s` con un costo máximo `presupuesto`.
6. Dos caminos pueden empatar en costo. En el grafo de la sección 9, ¿qué vértices tienen dos caminos más cortos distintos desde `0`? ¿Cuál de los dos elige `shortest_path_tree`, y por qué?
7. Explica con tus palabras por qué `shortest_path_lengths` necesita `pq.update`. ¿Qué habría que hacer si la cola no permitiera cambiar una clave?

In [4]:
import sys
sys.path.append(r'C:\Users\fator\Desktop\Proyectos1\Estructura_Datos\2026_02')

In [6]:
import inspect
from goodrich.ch14.shortest_paths import *

In [8]:
print(inspect.getsource(shortest_path_lengths))

def shortest_path_lengths(g, src):
  """Compute shortest-path distances from src to reachable vertices of g.

  Graph g can be undirected or directed, but must be weighted such that
  e.element() returns a numeric weight for each edge e.

  Return dictionary mapping each reachable vertex to its distance from src.
  """
  d = {}                                        # d[v] is upper bound from s to v
  cloud = {}                                    # map reachable v to its d[v] value
  pq = AdaptableHeapPriorityQueue()             # vertex v will have key d[v]
  pqlocator = {}                                # map from vertex to its pq locator

  # for each vertex v of the graph, add an entry to the priority queue, with
  # the source having distance 0 and all others having infinite distance
  for v in g.vertices():
    if v is src:
      d[v] = 0
    else:
      d[v] = float('inf')                       # syntax for positive infinity
    pqlocator[v] = pq.add(d[v], v)              # save